In [3]:
import duckdb
import pandas as pd

conn = duckdb.connect('../data/churn_database.duckdb')

In [4]:
query1 = """
SELECT 
    COUNT(*) AS total_customers,
    SUM(churn) AS churned_customers,
    (COUNT(*) - SUM(churn)) AS retained_customers,
    ROUND(AVG(churn) * 100, 2) AS churn_rate_pct,
    ROUND((1 - AVG(churn)) * 100, 2) AS retention_rate_pct
FROM customers;
"""
df_overall = conn.execute(query1).df()
display(df_overall)


,total_customers,churned_customers,retained_customers,churn_rate_pct,retention_rate_pct
0,1000000,99227.0,900773.0,9.92,90.08


In [5]:
query2 = """
SELECT 
    contract,
    COUNT(*) as total_customers,
    SUM(churn) as churned_customers,
    ROUND(AVG(churn) * 100, 2) as churn_rate_pct
FROM customers
GROUP BY contract
ORDER BY churn_rate_pct DESC;
"""
df_contract = conn.execute(query2).df()
display(df_contract)


,contract,total_customers,churned_customers,churn_rate_pct
0,month_to_month,19992,5299.0,26.51
1,one_year,550468,69651.0,12.65
2,two_year,429540,24277.0,5.65


In [6]:
query3 = """
SELECT 
    CASE WHEN churn = 1 THEN 'Churned' ELSE 'Retained' END as status,
    ROUND(AVG(tenure), 1) as avg_tenure_months,
    ROUND(AVG(totalcharges), 2) as avg_lifetime_value
FROM customers
GROUP BY churn;
"""
df_lifetime = conn.execute(query3).df()
display(df_lifetime)


,status,avg_tenure_months,avg_lifetime_value
0,Retained,22.5,1847.20
1,Churned,21.6,1747.65


In [7]:
query4 = """
SELECT 
    contract,
    payment_method,
    tenure_group,
    COUNT(*) as customer_count,
    ROUND(AVG(churn) * 100, 2) as churn_rate_pct
FROM customers
GROUP BY contract, payment_method, tenure_group
HAVING customer_count > 1000 -- Filter out tiny, noisy groups
ORDER BY churn_rate_pct DESC
LIMIT 5;
"""
df_high_risk = conn.execute(query4).df()
display(df_high_risk)


,contract,payment_method,tenure_group,customer_count,churn_rate_pct
0,month_to_month,credit_card,New,2859,31.55
1,month_to_month,mailed_check,New,1254,29.98
2,month_to_month,bank_transfer,New,2045,29.73
3,month_to_month,electronic_check,New,2091,28.36
4,month_to_month,electronic_check,Mid,1165,25.58


In [8]:
conn.close()